# Training Series: Introduction to eXtended Discontinuous Galerkin Methods for Multi-Phase Flow Problems <br> - Hands-On Worksheets

In [7]:
#r "../BoSSSpad/BoSSSpad.dll"
//#r "C:\Program Files (x86)\FDY\BoSSS\bin\Release\net5.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Databases loaded: 
Capacity: 0
Count: 0



Error: System.ApplicationException: Already called.
   at BoSSS.Application.BoSSSpad.BoSSSshell.InitTraceFile() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 217
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 104
   at Submission#7.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

## Problem statement: transient incompressible two-phase flow

The flow within the computational domain $\Omega = \mathfrak{A} \ {\dot\cup} \ \mathfrak{I} \ {\dot\cup} \ \mathfrak{B}, \Omega \in R^2$ is described by the instationary incompressible Navier-Stokes equations 
>$$ \rho_f\left(\frac{\partial \vec{u}}{\partial t}+ \nabla \cdot \left( \vec{u} \otimes \vec{u} \right) \right) = \nabla \cdot \left( -p\mathbf{I} + \mu_f \left( \nabla \vec{u} + \nabla \vec{u}^{\textrm{T}} \right) \right) = \rho_f \vec{f} \quad \textrm{in}\ \Omega \setminus \mathfrak{I} $$

and the continuity equation
>$$ \nabla \cdot \vec{u} = 0 \quad \textrm{in}\ \Omega \setminus \mathfrak{I} $$

The jump conditions are given by
>$$ \llbracket -p \mathbf{I} + \mu_f (\nabla \vec{u} + \nabla \vec{u}^{\textrm{T}}) \rrbracket \vec{n}_{\mathfrak{I}} = \sigma \kappa \vec{n}_{\mathfrak{I}}  $$
>$$ \llbracket \vec{u} \rrbracket  = \vec{0}  $$

At the boundary $\partial \Omega = \partial \Omega_\textrm{D} \cup \partial \Omega_\textrm{N}$ the corresponding boundary conditions read 
>$$ \vec{u} = \vec{u}_\textrm{D} \quad \textrm{on}\ \partial \Omega_\textrm{D} $$
>$$ -p \vec{n}_{\partial \Omega} + \mu_f \left( \nabla \vec{u} + \nabla \vec{u}^{\textrm{T}} \right) \vec{n}_{\partial \Omega} = - p_\textrm{ext} \vec{n}_{\partial \Omega} \quad \textrm{on}\ \partial \Omega_\textrm{N} $$

In the equations above 
 - $\vec{u}(\vec{x}, t)$ is the velocity vector 
 - $p(\vec{x}, t)$ the pressure
 - and $\vec{f}(\vec{x})$ some volume force, e.g. gravity. 
 - The fluid density is denoted by $\rho_f$ and
 - $\mu_f=\rho_f \cdot \nu_f$ is the dynamic viscosity of the fluid.
 - $\sigma$ denotes the surface tension force and
 - $\kappa$ the mean curvature of the interface $\mathfrak{I}$
 



## Setting up test cases

In [2]:
using System.IO;
using BoSSS.Solution.NSECommon;

In [10]:
// remove old plot files
public static void DeletePlotFilesInCurrentDirectory() {
    var dir = new DirectoryInfo(Directory.GetCurrentDirectory());
    Console.Write("rm");
    foreach (var pltFile in dir.GetFiles("XNSE-*")) {
        Console.Write(" " + pltFile.Name);
        pltFile.Delete();
    }
    foreach (var resFile in dir.GetFiles("residual-L2.*")) {
        Console.Write(" " + resFile.Name);
        resFile.Delete();
    }
    Console.WriteLine(";");
}

### Test case 1 - static droplet 

In [ ]:
XNSE_Control C1 = new XNSE_Control();

// database and saving options
C1.DbPath = null;
C1.savetodb = false;
C1.SessionName = "StatciDroplet";
C1.ImmediatePlotPeriod = 1;
C1.SuperSampling = 3;

// Physical parameters
C1.PhysicalParameters.rho_A = 1.0;
C1.PhysicalParameters.rho_B = 1.0;
C1.PhysicalParameters.mu_A = 1.0;
C1.PhysicalParameters.mu_B = 1.0;
C1.PhysicalParameters.Sigma = 1.0;

C1.PhysicalParameters.IncludeConvection = false;    // false -> Stokes
C1.PhysicalParameters.Material = true;

// DG degree
C1.SetDGdegree(3);

// computational grid
C1.GridFunc = delegate {

    double size = 1.0;
    int gridResolution = 8;
    double[] _Nodes = GenericBlas.Linspace(-size, size, gridResolution * 2 + 1);
    var grd = Grid2D.Cartesian2DGrid(_Nodes, _Nodes, BoSSS.Foundation.Grid.RefElements.CellType.Square_Linear);

    grd.DefineEdgeTags(delegate (double[] _X) {
        double x = _X[0];
        double y = _X[1];

    if(Math.Abs(y - (-size)) < 1.0e-6       // bottom
        || Math.Abs(y - (+size)) < 1.0e-6   // top
        || Math.Abs(x - (-size)) < 1.0e-6   // left
        || Math.Abs(x - (size)) < 1.0e-6)   // right
        return IncompressibleBcType.Wall.ToString();
    else
        throw new ArgumentOutOfRangeException();
    });

    return grd;
};

// Set initial level-set, resp. interface position
C1.AddInitialValue("Phi", new Formula("X => ((X[0]).Pow2() + X[1].Pow2()).Sqrt() - 0.5", TimeDep: false));  // signed-distance
//C1.AddInitialValue("Phi", new Formula("X => ((X[0]).Pow2() + X[1].Pow2()) - (0.5).Pow(2)", TimeDep: false));  // quadratic


// Time stepping
C1.TimesteppingMode = AppControl._TimesteppingMode.Steady;
C1.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
C1.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.None;
C1.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.None;


Note: Since the solver may be executed in an external program, the control object has to be saved in a file. 
For lots of complicated objects, especially for delegates, C\# does not support serialization (converting the object into a form that can be saved on disk, or transmitted over a network), so a workaround is needed.
This is achieved e.g. by the **Formula** object, where a C\#-formula is saved as a string.

In [12]:
// remove old plot files
DeletePlotFilesInCurrentDirectory();

rm;


In [13]:
C1.Run();

Grid Edge Tags changed.
Session ID: 00000000-0000-0000-0000-000000000000, DB path: 'EMPTY'
Session directory 'EMPTY\sessions\00000000-0000-0000-0000-000000000000'.
Grid repartitioning method: METIS
Grid repartitioning options: 
Number of cell Weights: 0
Going with agglomeration threshold: 0.1
Linearization hint: AdHoc
=============== Operator Configuration ===============
     isGravity                     :[ ]
     isVolForce                    :[ ]
     isTransport                   :[ ]
     isViscous                     :[x]
     isPressureGradient            :[x]
     isInterfaceSlip               :[ ]
     isContinuity                  :[x]
     isMovingMesh                  :[ ]
     isMatInt                      :[x]
     isPInterfaceSet               :[ ]
     isImmersedBoundary            :[ ]
     withPressureDissipation       :[ ]
=============== Linear Solver Configuration ===============
     Solvercode                    :Sparse direct solver PARDISO
=============== Nonl

### Test case 2 - droplet in channel flow

In [13]:
XNSE_Control C2 = new XNSE_Control();

// database and saving options
C2.DbPath = null;
C2.savetodb = false;
C2.SessionName = "ChannelFlow";
C2.ImmediatePlotPeriod = 1;
C2.SuperSampling = 2;

// Physical parameters
C2.PhysicalParameters.rho_A = 1.0;
C2.PhysicalParameters.rho_B = 1.0;
C2.PhysicalParameters.mu_A = 1.0;
C2.PhysicalParameters.mu_B = 0.01;
C2.PhysicalParameters.Sigma = 0.1;

C2.PhysicalParameters.IncludeConvection = true;
C2.PhysicalParameters.Material = true;

// DG degree
C2.SetDGdegree(3);

// computational grid
C2.GridFunc = delegate {
    int gridResolution = 4;

    double Length = 5.0;
    double Height = 2.0;

    double[] _xNodes = GenericBlas.Linspace(0, Length, gridResolution * 5 + 1);
    double[] _yNodes = GenericBlas.Linspace(-Height/2.0, Height/2.0, gridResolution * 2 + 1);

    var grd = Grid2D.Cartesian2DGrid(_xNodes, _yNodes, BoSSS.Foundation.Grid.RefElements.CellType.Square_Linear);

    grd.DefineEdgeTags(delegate (double[] _X) {
        double x = _X[0];
        double y = _X[1];

    if(Math.Abs(y - (-Height/2.0)) < 1.0e-6)
        // bottom
        return IncompressibleBcType.Wall.ToString() + "_bottom";

    if(Math.Abs(y - (+Height/2.0)) < 1.0e-6)
        // top
        return IncompressibleBcType.Wall.ToString()+ "_top";


    if(Math.Abs(x - (0.0)) < 1.0e-6)
        // left
        return IncompressibleBcType.Velocity_Inlet.ToString();

    if(Math.Abs(x - (Length)) < 1.0e-6)
        // right
        return IncompressibleBcType.Pressure_Outlet.ToString();

        throw new ArgumentOutOfRangeException();
    });

    return grd;
};

// set boundary conditions
C2.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX", new Formula("X => 1 - X[1] * X[1]", TimeDep: false));

// Set Initial Conditions
C2.AddInitialValue("VelocityX", new Formula("X => 0.0", TimeDep: false));

// Set initial level-set, resp. interface position
C2.AddInitialValue("Phi", new Formula("X => ((X[0] - 1.0).Pow2() + X[1].Pow2()).Sqrt() - 0.5", TimeDep: false));  // signed-distance

// Solver Options
C2.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C2.NonLinearSolver.ConvergenceCriterion = 1e-9;
C2.NonLinearSolver.MaxSolverIterations = 50; 

// Time stepping
C2.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C2.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
C2.Timestepper_LevelSetHandling = BoSSS.Solution.XdgTimestepping.LevelSetHandling.LieSplitting;
C2.Option_LevelSetEvolution = BoSSS.Solution.LevelSetTools.LevelSetEvolution.FastMarching;

// Adaptive mesh refinement
C2.AdaptiveMeshRefinement = true;
C2.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = 1 });
C2.AMR_startUpSweeps = 1;

C2.NoOfTimesteps = 100;
C2.dtFixed = 0.01;


In [14]:
// remove old plot files
DeletePlotFilesInCurrentDirectory();

rm XNSE-0.0.plt XNSE-0.1.plt XNSE-0.plt XNSE-1.plt residual-L2.2025May06_10-37-25.txt

Error: System.IO.IOException: The process cannot access the file 'd:\BoSSS-experimental\public\doc\TrainingSeries\introXDG\residual-L2.2025May06_10-37-25.txt' because it is being used by another process.
   at System.IO.FileSystem.DeleteFile(String fullPath)
   at System.IO.FileInfo.Delete()
   at Submission#10.DeletePlotFilesInCurrentDirectory()
   at Submission#14.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [15]:
C2.Run();

Grid Edge Tags changed.
Session ID: 00000000-0000-0000-0000-000000000000, DB path: 'EMPTY'
Session directory 'EMPTY\sessions\00000000-0000-0000-0000-000000000000'.
Grid repartitioning method: METIS
Grid repartitioning options: 
Number of cell Weights: 0
Going with agglomeration threshold: 0.1
Linearization hint: AdHoc
=============== Operator Configuration ===============
     isGravity                     :[ ]
     isVolForce                    :[ ]
     isTransport                   :[x]
     isViscous                     :[x]
     isPressureGradient            :[x]
     isInterfaceSlip               :[ ]
     isContinuity                  :[x]
     isMovingMesh                  :[ ]
     isMatInt                      :[x]
     isPInterfaceSet               :[ ]
     isImmersedBoundary            :[ ]
     withPressureDissipation       :[ ]
=============== Linear Solver Configuration ===============
     Solvercode                    :Sparse direct solver PARDISO
=============== Nonl

# Workflow Management and Meta Job Scheduler (BONUS)

This section illustrates how computations can be executed on large 
(or small) compute servers, aka.
high-performance-computers (HPC), aka. compute clusters, aka. supercomputers, etc.

This means, the computation is not executed on the local workstation 
(or laptop) but on some other computer.
This approach is particulary handy for large computations, which run
for multiple hours or days, since a user can 
e.g. shutdown or restart his personal computer without killing the compute job. 

*BoSSS* features a set of classes and routines 
(an API, application programing interface) for communication with 
compute clusters. This is especially handy for **scripting**,
e.g. for parameter studies, where dozens of computations 
have to be started and monitored.

### Batch Processing

First, we have to select a *batch system* (aka.*execution queue*, aka. *queue*) that we want to use.
Batch systems are a common approach to organize workloads (aka. compute jobs)
on compute clusters.
On such systems, a user typically does **not** starts a simulation manually/interactively.
Instead, he specifies a so-called *compute job*. The *scheduler* 
(i.e. the batch system) collects 
compute jobs from all users on the compute cluster, sorts them according to 
some priority and puts the jobs into some queue, also called *batch*.
The jobs in the batch are then executed in order, depending on the 
available hardware and the scheduling policies of the system.

The *BoSSS* API provides front-ends (clients) for the following 
batch system software:

- `BoSSS.Application.BoSSSpad.SlurmClient` for the 
Slurm Workload Manager (very prominent on Linux HPC systems)
- `BoSSS.Application.BoSSSpad.MsHPC2012Client`
for the Microsoft HPC Pack 2012 and higher
- `BoSSS.Application.BoSSSpad.MiniBatchProcessorClient` for the 
mini batch processor, a minimalistic, *BoSSS*-internal batch system which mimics 
a supercomputer batch system on the local machine.

A list of clients for various batch systems, which are loaded at the 
`Init()` command can be configured through the  
`~/.BoSSS/etc/BatchProcessorConfig.json`-file.
If this file is missing, a default setting, containing a 
mini batch processor, is initialized. 

The list of all execution queues can be accessed through:

In [15]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @D:\local\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"
2,BoSSS.Application.BoSSSpad.SlurmClient,"SlurmClient Lb2-specialPrj : nu39gihu@lcluster17.hrz.tu-darmstadt.de, Slurm account: special00006"


### Note on the Mini Batch Processor:
The batch processor for local jobs can be started separately (by launching
`MiniBatchProcessor.exe` or `dotnet MiniBatchProcessor.dll`), **which is the preferred option**.
Alternatively, it can be started from Jupyter Notebook; it depends on the operating system, whether the 
`MiniBatchProcessor.exe` is terminated with the notebook kernel, or not.
If no mini-batch-processor is running, it is started (hopefully) upon Job activation.

## Initializing the workflow management

In order to use the workflow management, 
the very first thing we have to do is to initialize it by defineing 
a **project name**, here it is `IntroXDG`. 
This is used to generate names for the compute jobs and to 
identify sessions in the database:

In [16]:
BoSSSshell.WorkflowMgm.Init("IntroXDG");

Project name is set to 'IntroXDG'.
Default Execution queue is chosen for the database.
Creating database 'D:\local\IntroXDG'.


For this project, the default execution queue is set to:

In [17]:
GetDefaultQueue()

DeploymentBaseDirectory,D:\local\binaries
DeployRuntime,False
RuntimeLocation,<null>
Name,LocalPC
DotnetRuntime,dotnet
BatchInstructionDir,<null>
AllowedDatabasesPaths,[ D:\local\ ]


## Activation and Monitoring of the the Job

Finally, we are ready to deploy the job at the batch processor;
In a usual work flow scenario, we **do not** want to (re-) submit the 
job every time we run the worksheet -- usually, one wants to run a job once.

The concept to overcome this problem is job activation. If a job is 
activated, the meta job manager first checks the databases and the batch 
system, if a job with the respective name and project name is already 
submitted. Only if there is no information that the job was ever submitted
or started anywhere, the job is submitted to the respective batch system.

In [31]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();  // check correlation by name
//BoSSSshell.WorkflowMgm.SetEqualityBasedSessionJobControlCorrelation();

First, a `Job` -object is created from the control object:

In [18]:
C1.savetodb = true;

In [19]:
var JobLocal = C1.CreateJob();

This job is not activated yet, it can still be configured:

In [20]:
JobLocal.Status

PreActivation

### Starting the compute Job

One can change e.g. the number of MPI processes:

In [21]:
JobLocal.NumberOfMPIProcs = 2;

Note that these jobs are desigend to be **persistent**:
This means the computation is only started 
**once for a given control object**, no matter how often the worksheet
is executed. 

Such a behaviour is useful for expensive simulations, which run on HPC
servers over days or even weeks. The user (you) can close the worksheet
and maybe open and execute it a few days later, and he can access
the original job which he submitted a few days ago (maybe it is finished
now).

Then, the job is activated, resp. submitted, resp. deployed 
to one batch system.
If job persistency is not wanted, traces of the job can be removed 
on request during activation, causing a fresh job deployment at the
batch system:

In [22]:
JobLocal.Activate();  // execute the job in the default execution queue
//JobLocal.Activate(ExecutionQueues[4]);  // execute the job e.g. in queue 4

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job UnnamedJob_1 ... 
Creating database '\\dc3\userspace\smuda\hpccluster\IntroXDG'.
Creating database '\\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases\IntroXDG'.
Set Database: { Session Count = 0; Grid Count = 0; Path = D:\local\IntroXDG }
Control object contains grid function. Trying to Serialize the grid...
Grid Edge Tags changed.
Using grid 9c670782-23da-413c-a006-88f96d8cd6f5 at D:\local\IntroXDG.
Control object modified.


Deploying executables and additional files ...
Deployment directory: D:\local\binaries\IntroXDG-XNSE_Solver2025Apr24_100158.627788
copied 43 files.
   written file: control.obj
deployment finished.
Starting mini batch processor in external process...
Started mini batch processor on local machine, process id is 15416.
started.


All jobs can be listed using the workflow management:

In [23]:
BoSSSshell.WorkflowMgm.AllJobs

#0: UnnamedJob_1: FinishedSuccessful (MiniBatchProcessor client  LocalPC @D:\local\binaries)	UnnamedJob_1: FinishedSuccessful (MiniBatchProcessor client  LocalPC @D:\local\binaries)


Check the present job status:

In [24]:
JobLocal.Status

FinishedSuccessful

### Evaluation of Job

We examine the output and error stream of the job:
This directly accesses the *\tt stdout*-redirection of the respective job
manager, which may contain a bit more information than the 
`Stdout`-copy in the session directory.

In [25]:
JobLocal.Stdout

Session ID: 5390a035-7cec-4671-8ef3-388104b48499, DB path: 'D:\local\IntroXDG'
Session directory 'D:\local\IntroXDG\sessions\5390a035-7cec-4671-8ef3-388104b48499'.
Grid repartitioning method: METIS
Grid repartitioning options: 
Number of cell Weights: 0
noOfPartitioningsToChooseFrom = 1
Cells per rank: [128, 128]
Going with agglomeration threshold: 0.1
Linearization hint: AdHoc
=============== Operator Configuration ===============
     isGravity                     :[ ]
     isVolForce                    :[ ]
     isTransport                   :[ ]
     isViscous                     :[x]
     isPressureGradient            :[x]
     isInterfaceSlip               :[ ]
     isContinuity                  :[x]
     isMovingMesh                  :[ ]
     isMatInt                      :[x]
     isPInterfaceSet               :[ ]
     isImmersedBoundary            :[ ]
     withPressureDissipation       :[ ]
=============== Linear Solver Configuration ===============
     Solvercode         

Additionally we display the error stream and hope that it is empty:

In [26]:
JobLocal.Stderr

We can also obtain the session 
which was stored during the execution of the job:

In [27]:
var Sloc = JobLocal.LatestSession;
Sloc

IntroXDG	UnnamedJob_1	4/24/2025 10:02:33 AM	5390a035...

## Exporting Plots

Each run of the solver corresponds to one **session** in the database. A session is basically a collection of information on the entire solver run, i.e. the simulation result, input and solver settings as well as meta-data such as computer and data and time.

Since in this tutorial only one solver run was executed, there is only one session in the Workflow Management (`wmg` is just an alias for `BoSSSshell.WorkflowMgm.`):

In [28]:
wmg.Sessions

#0: IntroXDG	UnnamedJob_1	4/24/2025 10:02:33 AM	5390a035...


We select the first (and only) session and create an **export instruction** object.
The **supersampling** setting increases the output resolution. 

In [29]:
var outPath = wmg.Sessions[0].Export().WithSupersampling(2).Do();

Starting export process... Data will be written to the directory: C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\IntroXDG__UnnamedJob_1__5390a035-7cec-4671-8ef3-388104b48499


On the respective directory (see output above) one should finally find plot-files which than can be used
for further post processing in third-party software such as Paraview, LLNL Visit or Tecplot.

The `Do()` command returns the location of the output files:

In [30]:
outPath

C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\IntroXDG__UnnamedJob_1__5390a035-7cec-4671-8ef3-388104b48499